<a href="https://colab.research.google.com/github/kamal-gavel/PCA-Based-Signal-to-Noise-Decomposition-of-Financial-Sentiment-and-Market-Features/blob/main/PCA_Based_Signal_to_Noise_Decomposition_of_Financial_Sentiment_and_Market_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Good — now we connect theory → your dataset → actual projection matrix (P) in one clean pipeline.

Below is a single-cell, production-ready code that:

✔ Cleans your dataset
✔ Removes non-numeric (date issue fix)
✔ Standardizes data
✔ Computes PCA
✔ Builds Projection Matrix
𝑃
=
𝑈
𝑈
𝑇
P=UU
T

✔ Computes projection
𝑃
𝑏
Pb
✔ Computes error
𝑏
−
𝑃
𝑏
b−Pb
✔ Shows how much signal vs noise you have
"""

In [3]:
# ===============================================
# PROJECTION MATRIX ON YOUR DATASET (ROBUST VERSION)
# ===============================================

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ===============================================
# 1. LOAD YOUR DATA
# ===============================================
file_path = "panel_dataset_extended_horizons.csv"
df = pd.read_csv(file_path)

print("Original Dataset shape:", df.shape)
print(df.head())

# ===============================================
# 2. HANDLE DATE / NON-NUMERIC COLUMNS
# ===============================================

# keep date separately (important for later alignment)
date_cols = [col for col in df.columns if 'date' in col.lower()]
dates = df[date_cols] if len(date_cols) > 0 else None

# keep only numeric columns
df_numeric = df.select_dtypes(include=[np.number])

# fill missing values (important for PCA)
df_numeric = df_numeric.ffill().bfill()

print("Numeric Dataset shape:", df_numeric.shape)

# ===============================================
# 3. FEATURE MATRIX
# ===============================================
X = df_numeric.values

# ===============================================
# 4. STANDARDIZATION
# ===============================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ===============================================
# 5. PCA (AUTO COMPONENT SELECTION OPTION)
# ===============================================

# Option 1: Fixed components
# n_components = 10

# Option 2 (Recommended): keep 95% variance
pca = PCA(n_components=0.95)

X_pca = pca.fit_transform(X_scaled)
U = pca.components_.T

print("\nSelected Components:", U.shape[1])

# ===============================================
# 6. PROJECTION MATRIX
# ===============================================
P = U @ U.T

print("Projection Matrix shape:", P.shape)

# ===============================================
# 7. PROJECT DATA
# ===============================================
X_proj = X_scaled @ P

# ===============================================
# 8. RESIDUAL (NOISE)
# ===============================================
X_error = X_scaled - X_proj

# ===============================================
# 9. SIGNAL vs NOISE ANALYSIS
# ===============================================
signal_power = np.linalg.norm(X_proj)
noise_power = np.linalg.norm(X_error)

print("\n===================================")
print("📊 PROJECTION ANALYSIS")
print("===================================")
print("Signal Power:", signal_power)
print("Noise Power:", noise_power)
print("Signal/Noise Ratio:", signal_power / (noise_power + 1e-9))

# ===============================================
# 10. EXPLAINED VARIANCE
# ===============================================
explained_var = pca.explained_variance_ratio_

print("\nExplained Variance Ratio:")
print(explained_var)
print("Total Explained Variance:", np.sum(explained_var))

# ===============================================
# 11. PROJECTION MATRIX VALIDATION
# ===============================================
symmetry_error = np.linalg.norm(P - P.T)
idempotent_error = np.linalg.norm(P @ P - P)

print("\n===================================")
print("🧠 PROJECTION MATRIX PROPERTIES")
print("===================================")
print("Symmetry Error:", symmetry_error)
print("Idempotent Error:", idempotent_error)

# ===============================================
# 12. RECONSTRUCTION ERROR
# ===============================================
reconstruction_error = np.mean((X_scaled - X_proj) ** 2)

print("\n===================================")
print("📉 RECONSTRUCTION ERROR")
print("===================================")
print("MSE:", reconstruction_error)

# ===============================================
# 13. OPTIONAL: SAVE RESULTS
# ===============================================
results_df = pd.DataFrame(X_proj, columns=df_numeric.columns)

if dates is not None:
    results_df = pd.concat([dates.reset_index(drop=True), results_df], axis=1)

save_path = "projected_dataset.csv"
results_df.to_csv(save_path, index=False)

print(f"\n✅ Saved projected dataset to: {save_path}")

# ===============================================
print("\n🚀 Projection Matrix Pipeline Completed")

Original Dataset shape: (1521, 34)
  final_date  avg_sentiment  news_volume  pos_ratio  neg_ratio  \
0  5/22/2018       0.273667            3   0.666667        0.0   
1  5/31/2018       0.777900            1   1.000000        0.0   
2   7/4/2018       0.435900            1   1.000000        0.0   
3  8/16/2018      -0.906000            1   0.000000        1.0   
4  8/17/2018       0.382100            1   1.000000        0.0   

   sentiment_pressure     company        date        Close  return_t+1  ...  \
0            0.666667  apollohosp  2018-05-22   968.299988    0.019984  ...   
1            1.000000  apollohosp  2018-05-31   950.349976   -0.024465  ...   
2            1.000000  apollohosp  2018-07-04  1081.900024   -0.031195  ...   
3           -1.000000  apollohosp  2018-08-16  1140.500000    0.042701  ...   
4            1.000000  apollohosp  2018-08-17  1189.199951   -0.070930  ...   

   return_3  volatility_5  sentiment_std         ma10         ma20  \
0 -0.174404      0.1166

In [4]:
#👉 How much signal comes ONLY from sentiment features
# ===============================================
# SENTIMENT-ONLY SIGNAL ANALYSIS
# ===============================================

# define sentiment columns
sentiment_cols = [
    'avg_sentiment', 'news_volume', 'pos_ratio',
    'neg_ratio', 'sentiment_pressure', 'sentiment_std'
]

df_sent = df[sentiment_cols].copy()

# clean
df_sent = df_sent.ffill().bfill()

X_sent = df_sent.values

# standardize
X_sent_scaled = StandardScaler().fit_transform(X_sent)

# PCA
pca_sent = PCA(n_components=0.95)
X_sent_pca = pca_sent.fit_transform(X_sent_scaled)

U_sent = pca_sent.components_.T
P_sent = U_sent @ U_sent.T

# projection
X_sent_proj = X_sent_scaled @ P_sent
X_sent_error = X_sent_scaled - X_sent_proj

# signal vs noise
signal_sent = np.linalg.norm(X_sent_proj)
noise_sent = np.linalg.norm(X_sent_error)

print("\n===================================")
print("🧠 SENTIMENT SIGNAL ANALYSIS")
print("===================================")
print("Signal:", signal_sent)
print("Noise:", noise_sent)
print("SNR:", signal_sent / (noise_sent + 1e-9))


🧠 SENTIMENT SIGNAL ANALYSIS
Signal: 93.19592557599277
Noise: 20.988555358433164
SNR: 4.440321116913199
